# Home Credit Proxy Notebook 05 — Benchmark & Registry

Mục tiêu của notebook này:

1. **Đọc artifact từ notebook 4** thay vì train lại model.
2. **Benchmark stage A / B / C** theo góc nhìn business + operational policy.
3. **Đề xuất cutoff / action zone** cho từng stage theo đúng vai trò:
   - **A** = starter-offer / onboarding recommendation
   - **B** = full-decision candidate
   - **C** = guardrail / collection prioritization
4. **Tạo model registry / handoff package** cho notebook tiếp theo hoặc Streamlit/API.

> Ghi chú:
> - `score_proba` **không phải calibrated PD**.
> - `score_raw` / `score_300_900` chỉ dùng cho **ranking / policy exploration**.
> - Higher score = **higher risk** trong flow hiện tại.


In [ ]:
# =========================================================
# 0. Imports
# =========================================================
from pathlib import Path
import json
import math
import warnings

import joblib
import numpy as np
import polars as pl

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(120)


In [ ]:
# =========================================================
# 1. Locate notebook 4 artifact directory
# =========================================================
DEFAULT_INPUT_DIR_NB4 = Path("/kaggle/input/notebooks/hongvittrnh/notebook-4-0804-revised-a/homecredit_proxy_notebook_04_business")

candidate_roots = [
    DEFAULT_INPUT_DIR_NB4,
    Path("/kaggle/working/homecredit_proxy_notebook_04_business"),
    Path("/kaggle/working"),
    Path("/mnt/data"),
    Path.cwd(),
]

required_markers = [
    "stage_model_metrics.csv",
    "stage_business_readiness.csv",
    "product_scoring_manifest.json",
    "unified_stage_inference_pack.joblib",
]

INPUT_DIR_NB4 = None

for root in candidate_roots:
    if root.is_dir() and all((root / f).exists() for f in required_markers):
        INPUT_DIR_NB4 = root
        break

if INPUT_DIR_NB4 is None:
    # recursive fallback
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data"), Path.cwd()]:
        if not root.exists():
            continue
        for p in root.rglob("stage_model_metrics.csv"):
            parent = p.parent
            if all((parent / f).exists() for f in required_markers):
                INPUT_DIR_NB4 = parent
                break
        if INPUT_DIR_NB4 is not None:
            break

if INPUT_DIR_NB4 is None:
    raise FileNotFoundError(
        "Cannot locate notebook 4 artifact folder. "
        "Expected files: stage_model_metrics.csv, stage_business_readiness.csv, "
        "product_scoring_manifest.json, unified_stage_inference_pack.joblib"
    )

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_05_benchmark_registry")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR_NB4:", INPUT_DIR_NB4)
print("WORK_DIR     :", WORK_DIR)


In [ ]:
# =========================================================
# 2. Load notebook 4 core artifacts
# =========================================================
metrics_df = pl.read_csv(INPUT_DIR_NB4 / "stage_model_metrics.csv")
readiness_df = pl.read_csv(INPUT_DIR_NB4 / "stage_business_readiness.csv")
policy_summary_df = pl.read_csv(INPUT_DIR_NB4 / "stage_product_policy_summary.csv")
feature_profile_df = (
    pl.read_csv(INPUT_DIR_NB4 / "stage_feature_profile.csv")
    if (INPUT_DIR_NB4 / "stage_feature_profile.csv").exists()
    else pl.DataFrame()
)
feature_stability_df = (
    pl.read_csv(INPUT_DIR_NB4 / "stage_feature_stability.csv")
    if (INPUT_DIR_NB4 / "stage_feature_stability.csv").exists()
    else pl.DataFrame()
)

with open(INPUT_DIR_NB4 / "product_scoring_manifest.json", "r", encoding="utf-8") as f:
    product_manifest = json.load(f)

inference_pack = joblib.load(INPUT_DIR_NB4 / "unified_stage_inference_pack.joblib")

stage_valid_scored = {}
stage_product_scored = {}

for stage in ["A", "B", "C"]:
    valid_path = INPUT_DIR_NB4 / f"stage_{stage.lower()}_valid_scored.parquet"
    product_path = INPUT_DIR_NB4 / f"stage_{stage.lower()}_product_scored.parquet"
    if valid_path.exists():
        stage_valid_scored[stage] = pl.read_parquet(valid_path)
    if product_path.exists():
        stage_product_scored[stage] = pl.read_parquet(product_path)

print("Loaded valid scored stages:", sorted(stage_valid_scored.keys()))
print("Loaded product scored stages:", sorted(stage_product_scored.keys()))

display(metrics_df)
display(readiness_df)


In [ ]:
# =========================================================
# 3. Benchmark summary table
# =========================================================
benchmark_df = (
    metrics_df
    .join(
        readiness_df.select([
            "stage",
            "policy_mode",
            "ready_for_nb5_business_strict",
            "ready_for_product_scoring_api",
            "note",
        ]),
        on="stage",
        how="left",
    )
    .with_columns([
        pl.col("auc_valid").rank(method="ordinal", descending=True).alias("rank_auc_valid"),
        pl.col("ks_valid").rank(method="ordinal", descending=True).alias("rank_ks_valid"),
        pl.col("pr_auc_valid").rank(method="ordinal", descending=True).alias("rank_pr_auc_valid"),
        pl.col("n_features_kept").rank(method="ordinal", descending=True).alias("rank_features_kept"),
    ])
    .with_columns(
        (
            0.40 * pl.col("rank_auc_valid") +
            0.25 * pl.col("rank_ks_valid") +
            0.20 * pl.col("rank_pr_auc_valid") +
            0.15 * pl.col("rank_features_kept")
        ).alias("composite_rank_score")
    )
    .sort("stage")
)

display(benchmark_df)

champion_decision_df = (
    benchmark_df
    .filter(pl.col("policy_mode") == "full_decision")
    .sort(["ready_for_nb5_business_strict", "auc_valid", "ks_valid", "n_features_kept"], descending=[True, True, True, True])
)

champion_decision_stage = champion_decision_df["stage"][0] if champion_decision_df.height > 0 else None
print("Champion stage for decision-oriented policy:", champion_decision_stage)


In [ ]:
# =========================================================
# 4. Helper functions for cutoff / benchmark analysis
# =========================================================
TARGET_COL = "target"
SCORE_COL = "score_300_900"
RAW_SCORE_COL = "score_raw"
PROBA_COL = "score_proba"

def safe_div(num: float, den: float) -> float:
    return float(num / den) if den not in [0, 0.0, None] else float("nan")

def np_quantile_safe(arr: np.ndarray, q: float) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.quantile(arr, q))

def binary_metrics_from_threshold(df: pl.DataFrame, threshold: float, score_col: str = SCORE_COL) -> dict:
    pdf = df.select([TARGET_COL, score_col]).drop_nulls().to_pandas()
    if pdf.empty:
        return {}
    y = pdf[TARGET_COL].astype(int).to_numpy()
    s = pdf[score_col].astype(float).to_numpy()

    pred_high = (s >= threshold).astype(int)

    tp = int(((pred_high == 1) & (y == 1)).sum())
    fp = int(((pred_high == 1) & (y == 0)).sum())
    fn = int(((pred_high == 0) & (y == 1)).sum())
    tn = int(((pred_high == 0) & (y == 0)).sum())

    flagged = int(pred_high.sum())
    total = int(len(pred_high))
    bads = int(y.sum())
    goods = int(total - bads)

    return {
        "threshold": float(threshold),
        "n_cases": total,
        "n_flagged_high_risk": flagged,
        "flagged_rate": safe_div(flagged, total),
        "tp_bad_captured": tp,
        "fp_good_flagged": fp,
        "fn_bad_missed": fn,
        "tn_good_safe": tn,
        "capture_rate_bad": safe_div(tp, bads),
        "precision_flagged_bad_rate": safe_div(tp, tp + fp),
        "specificity_goods_kept": safe_div(tn, goods),
        "bad_rate_flagged": safe_div(tp, tp + fp),
        "bad_rate_unflagged": safe_div(fn, fn + tn),
    }

def top_k_capture_table(df: pl.DataFrame, stage: str, score_col: str = SCORE_COL) -> pl.DataFrame:
    use = (
        df.select([TARGET_COL, score_col])
        .drop_nulls()
        .sort(score_col, descending=True)
        .with_row_count("rank_idx")
    )
    if use.height == 0:
        return pl.DataFrame()

    total_rows = use.height
    total_bad = int(use[TARGET_COL].sum())

    rows = []
    for frac in [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30]:
        k = max(1, int(round(total_rows * frac)))
        top_df = use.head(k)
        captured_bad = int(top_df[TARGET_COL].sum())
        rows.append({
            "stage": stage,
            "top_fraction": frac,
            "n_cases_top": k,
            "captured_bad": captured_bad,
            "capture_rate_bad": safe_div(captured_bad, total_bad),
            "bad_rate_top": float(top_df[TARGET_COL].mean()) if top_df.height > 0 else float("nan"),
            "avg_score_top": float(top_df[score_col].mean()) if top_df.height > 0 else float("nan"),
        })
    return pl.DataFrame(rows)

def make_zone_from_thresholds(df: pl.DataFrame, review_threshold: float, reject_threshold: float, score_col: str = SCORE_COL) -> pl.DataFrame:
    return (
        df.with_columns(
            pl.when(pl.col(score_col) >= reject_threshold)
            .then(pl.lit("reject_or_intensive_review"))
            .when(pl.col(score_col) >= review_threshold)
            .then(pl.lit("manual_review"))
            .otherwise(pl.lit("approve_or_continue"))
            .alias("decision_zone")
        )
    )

def summarize_zone_table(df: pl.DataFrame, zone_col: str = "decision_zone", score_col: str = SCORE_COL) -> pl.DataFrame:
    return (
        df.group_by(zone_col)
        .agg([
            pl.len().alias("n_cases"),
            pl.mean(TARGET_COL).alias("bad_rate"),
            pl.mean(score_col).alias("avg_score_300_900"),
        ])
        .sort("avg_score_300_900", descending=True)
    )


In [ ]:
# =========================================================
# 5. Threshold benchmark grid by stage
# =========================================================
threshold_rows = []
capture_tables = []

for stage, df in stage_valid_scored.items():
    if SCORE_COL not in df.columns:
        continue

    scores = df[SCORE_COL].to_numpy()
    candidate_quantiles = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

    seen = set()
    for q in candidate_quantiles:
        thr = round(np_quantile_safe(scores, q), 6)
        if thr in seen or math.isnan(thr):
            continue
        seen.add(thr)

        row = binary_metrics_from_threshold(df, thr, score_col=SCORE_COL)
        if not row:
            continue

        row["stage"] = stage
        row["threshold_quantile"] = q
        threshold_rows.append(row)

    capture_tbl = top_k_capture_table(df, stage=stage, score_col=SCORE_COL)
    if capture_tbl.height > 0:
        capture_tables.append(capture_tbl)

threshold_benchmark_df = (
    pl.DataFrame(threshold_rows)
    .sort(["stage", "threshold"], descending=[False, False])
    if threshold_rows else pl.DataFrame()
)

capture_benchmark_df = (
    pl.concat(capture_tables, how="vertical")
    .sort(["stage", "top_fraction"])
    if len(capture_tables) > 0 else pl.DataFrame()
)

display(threshold_benchmark_df)
display(capture_benchmark_df)


In [ ]:
# =========================================================
# 6. Stage-specific policy recommendation
# =========================================================
policy_reco_rows = []
zone_summary_tables = {}

for stage, df in stage_valid_scored.items():
    if SCORE_COL not in df.columns:
        print(f"[WARN] Skip stage {stage}: missing {SCORE_COL}")
        continue

    use_df = df.select([c for c in [TARGET_COL, SCORE_COL] if c in df.columns]).drop_nulls()
    if use_df.height == 0:
        print(f"[WARN] Skip stage {stage}: no non-null rows for policy recommendation")
        continue

    readiness_rows = readiness_df.filter(pl.col("stage") == stage).to_dicts()
    readiness_row = readiness_rows[0] if len(readiness_rows) > 0 else {}
    policy_mode = readiness_row.get("policy_mode", "guardrail_only")

    scores = use_df[SCORE_COL].to_numpy()
    q70 = np_quantile_safe(scores, 0.70)
    q75 = np_quantile_safe(scores, 0.75)
    q80 = np_quantile_safe(scores, 0.80)
    q85 = np_quantile_safe(scores, 0.85)
    q90 = np_quantile_safe(scores, 0.90)
    q95 = np_quantile_safe(scores, 0.95)

    # fallback nếu quantile bị nan do dữ liệu quá xấu / rỗng bất thường
    if math.isnan(q95):
        q95 = q90
    if math.isnan(q90):
        q90 = q85
    if math.isnan(q85):
        q85 = q80
    if math.isnan(q80):
        q80 = q75
    if math.isnan(q75):
        q75 = q70
    if math.isnan(q70):
        q70 = float(use_df[SCORE_COL].mean())

    if stage == "A":
        review_thr = q75
        reject_thr = q90
        zoned = (
            use_df.with_columns(
                pl.when(pl.col(SCORE_COL) >= reject_thr)
                .then(pl.lit("decline_or_manual_review"))
                .when(pl.col(SCORE_COL) >= review_thr)
                .then(pl.lit("manual_review"))
                .when(pl.col(SCORE_COL) >= q70)
                .then(pl.lit("starter_loan_small"))
                .otherwise(pl.lit("starter_loan_standard"))
                .alias("decision_zone")
            )
        )
        note = (
            "Stage A is thin-file / new-to-bank. "
            "Use zones as starter-offer recommendation, not strict approval PD."
        )

    elif stage == "B":
        review_thr = q75
        reject_thr = q90
        zoned = make_zone_from_thresholds(
            use_df,
            review_threshold=review_thr,
            reject_threshold=reject_thr,
            score_col=SCORE_COL,
        )
        note = (
            "Stage B is the primary decision candidate. "
            "Below review threshold can be used for safer approve/continue flow; "
            "middle zone for manual review; top zone for reject/intensive review experiments."
        )

    else:  # C
        review_thr = q80
        reject_thr = q95
        zoned = (
            use_df.with_columns(
                pl.when(pl.col(SCORE_COL) >= reject_thr)
                .then(pl.lit("intensive_collection_priority"))
                .when(pl.col(SCORE_COL) >= review_thr)
                .then(pl.lit("high_risk_review_priority"))
                .otherwise(pl.lit("monitor_or_standard_queue"))
                .alias("decision_zone")
            )
        )
        note = (
            "Stage C should be used for guardrail / collection prioritization, "
            "not hard underwriting approval."
        )

    zone_summary = summarize_zone_table(zoned)
    zone_summary_tables[stage] = zone_summary

    policy_reco_rows.append({
        "stage": stage,
        "policy_mode": policy_mode,
        "review_threshold_score_300_900": float(review_thr),
        "reject_threshold_score_300_900": float(reject_thr),
        "n_valid_rows": int(use_df.height),
        "avg_score_valid": float(use_df[SCORE_COL].mean()),
        "bad_rate_valid": float(use_df[TARGET_COL].mean()),
        "recommended_role": (
            "starter_offer" if stage == "A" else
            "decision_champion" if stage == "B" else
            "collection_guardrail"
        ),
        "note": note,
    })

policy_recommendation_df = pl.DataFrame(policy_reco_rows).sort("stage") if len(policy_reco_rows) > 0 else pl.DataFrame()
display(policy_recommendation_df)

for stage, tbl in zone_summary_tables.items():
    print(f"[ZONE SUMMARY] Stage {stage}")
    display(tbl)


In [ ]:
# =========================================================
# 7. Model registry table
# =========================================================
registry_rows = []

for stage in ["A", "B", "C"]:
    model_entry = inference_pack["stage_models"].get(stage, {})
    manifest_entry = product_manifest.get("stages", {}).get(stage, {})
    metrics_rows = metrics_df.filter(pl.col("stage") == stage).to_dicts()
    readiness_rows = readiness_df.filter(pl.col("stage") == stage).to_dicts()
    policy_rows = policy_recommendation_df.filter(pl.col("stage") == stage).to_dicts()

    m = metrics_rows[0] if len(metrics_rows) > 0 else {}
    r = readiness_rows[0] if len(readiness_rows) > 0 else {}
    p = policy_rows[0] if len(policy_rows) > 0 else {}

    registry_role = (
        "champion_decision_model" if stage == champion_decision_stage else
        "active_guarded_model" if bool(r.get("ready_for_product_scoring_api", False)) else
        "hold"
    )

    artifact_model = INPUT_DIR_NB4 / f"stage_{stage.lower()}_model.joblib"
    artifact_valid = INPUT_DIR_NB4 / f"stage_{stage.lower()}_valid_scored.parquet"
    artifact_product = INPUT_DIR_NB4 / f"stage_{stage.lower()}_product_scored.parquet"
    artifact_band = INPUT_DIR_NB4 / f"stage_{stage.lower()}_band_table.csv"

    registry_rows.append({
        "stage": stage,
        "registry_role": registry_role,
        "stage_name": manifest_entry.get("stage_name"),
        "policy_mode": r.get("policy_mode"),
        "ready_for_nb5_business_strict": r.get("ready_for_nb5_business_strict"),
        "ready_for_product_scoring_api": r.get("ready_for_product_scoring_api"),
        "n_features_kept": m.get("n_features_kept"),
        "auc_valid": m.get("auc_valid"),
        "ks_valid": m.get("ks_valid"),
        "pr_auc_valid": m.get("pr_auc_valid"),
        "review_threshold_score_300_900": p.get("review_threshold_score_300_900"),
        "reject_threshold_score_300_900": p.get("reject_threshold_score_300_900"),
        "recommended_role": p.get("recommended_role"),
        "recommended_use": manifest_entry.get("recommended_use"),
        "score_note": manifest_entry.get("score_note"),
        "artifact_model_path": str(artifact_model) if artifact_model.exists() else None,
        "artifact_valid_scored_path": str(artifact_valid) if artifact_valid.exists() else None,
        "artifact_product_scored_path": str(artifact_product) if artifact_product.exists() else None,
        "artifact_band_table_path": str(artifact_band) if artifact_band.exists() else None,
    })

registry_df = pl.DataFrame(registry_rows).sort("stage")
display(registry_df)


In [ ]:
# =========================================================
# 8. Registry manifest / handoff pack
# =========================================================
registry_manifest = {
    "registry_name": "homecredit_proxy_stage_registry_v1",
    "source_notebook_4_dir": str(INPUT_DIR_NB4),
    "work_dir_notebook_5": str(WORK_DIR),
    "global_note": (
        "Notebook 5 benchmark + registry package. "
        "score_proba is not calibrated PD. Higher score means higher risk. "
        "Stage A is starter-offer model, Stage B is main decision candidate, Stage C is guardrail/collection."
    ),
    "champion_decision_stage": champion_decision_stage,
    "stages": {},
}

for row in registry_rows:
    stage = row["stage"]
    registry_manifest["stages"][stage] = {
        "registry_role": row["registry_role"],
        "stage_name": row["stage_name"],
        "policy_mode": row["policy_mode"],
        "ready_for_nb5_business_strict": row["ready_for_nb5_business_strict"],
        "ready_for_product_scoring_api": row["ready_for_product_scoring_api"],
        "n_features_kept": row["n_features_kept"],
        "metrics": {
            "auc_valid": row["auc_valid"],
            "ks_valid": row["ks_valid"],
            "pr_auc_valid": row["pr_auc_valid"],
        },
        "recommended_thresholds": {
            "review_threshold_score_300_900": row["review_threshold_score_300_900"],
            "reject_threshold_score_300_900": row["reject_threshold_score_300_900"],
        },
        "recommended_role": row["recommended_role"],
        "recommended_use": row["recommended_use"],
        "artifacts": {
            "model": row["artifact_model_path"],
            "valid_scored": row["artifact_valid_scored_path"],
            "product_scored": row["artifact_product_scored_path"],
            "band_table": row["artifact_band_table_path"],
        },
    }

registry_pack = {
    "registry_manifest": registry_manifest,
    "inference_pack": inference_pack,
    "policy_recommendation_rows": policy_reco_rows,
}


In [ ]:
# =========================================================
# 9. Export notebook 5 artifacts
# =========================================================
saved_paths = []

benchmark_path = WORK_DIR / "stage_benchmark_summary.csv"
benchmark_df.write_csv(benchmark_path)
saved_paths.append(str(benchmark_path))

threshold_path = WORK_DIR / "stage_threshold_benchmark.csv"
threshold_benchmark_df.write_csv(threshold_path)
saved_paths.append(str(threshold_path))

capture_path = WORK_DIR / "stage_topk_capture_benchmark.csv"
capture_benchmark_df.write_csv(capture_path)
saved_paths.append(str(capture_path))

policy_path = WORK_DIR / "stage_policy_recommendation.csv"
policy_recommendation_df.write_csv(policy_path)
saved_paths.append(str(policy_path))

registry_path = WORK_DIR / "model_registry_table.csv"
registry_df.write_csv(registry_path)
saved_paths.append(str(registry_path))

manifest_path = WORK_DIR / "model_registry_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(registry_manifest, f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(manifest_path))

registry_pack_path = WORK_DIR / "model_registry_pack.joblib"
joblib.dump(registry_pack, registry_pack_path)
saved_paths.append(str(registry_pack_path))

for stage, tbl in zone_summary_tables.items():
    p = WORK_DIR / f"stage_{stage.lower()}_decision_zone_summary.csv"
    tbl.write_csv(p)
    saved_paths.append(str(p))

print("Saved notebook 5 artifacts:")
for p in saved_paths:
    print("-", p)


In [ ]:
# =========================================================
# 10. Final recommendation
# =========================================================
print("Champion decision stage:", champion_decision_stage)
print()

for row in registry_rows:
    print(f"[{row['stage']}] {row['stage_name']}")
    print(" - registry_role :", row["registry_role"])
    print(" - policy_mode   :", row["policy_mode"])
    print(" - auc_valid     :", row["auc_valid"])
    print(" - ks_valid      :", row["ks_valid"])
    print(" - n_features    :", row["n_features_kept"])
    print(" - recommended_role:", row["recommended_role"])
    print(" - review_threshold_score_300_900:", row["review_threshold_score_300_900"])
    print(" - reject_threshold_score_300_900:", row["reject_threshold_score_300_900"])
    print()

print("Notebook 5 conclusion:")
print("- Stage A: keep as starter-offer / onboarding recommendation model.")
print("- Stage B: use as primary decision benchmark / cutoff exploration stage.")
print("- Stage C: use as guardrail / collection prioritization stage.")
print("- Do not communicate score_proba as PD until calibration is done in a separate notebook / step.")
